# Normalized Drifting Model Trajectories

This notebook generates `fig:generative-drifting-model-trajectories`.  A drifting field moves model particles toward data particles with a kernel attraction.  The normalized version uses local kernel averages
$$
    B_\epsilon[\nu](x)=\frac{\int e^{-\|x-y\|^2/(2\epsilon)}(y-x)\,d\nu(y)}{\int e^{-\|x-y\|^2/(2\epsilon)}\,d\nu(y)},
$$
and subtracts a self-interaction term to avoid collapse.

In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd()
if ROOT.name == "notebooks-figures":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "notebooks-figures"))

from figure_style import (
    BLUE,
    RED,
    VIOLET,
    ORANGE,
    GRAY,
    LIGHT_GRAY,
    DIRAC_MARKER_SIZE,
    box_axes,
    draw_point_clouds,
    draw_transport_segments,
    figure_dir,
    interp_color,
    padded_limits,
    remove_axes,
    save_pdf,
    setup_matplotlib,
)

setup_matplotlib()

from matplotlib.collections import LineCollection

NAME = "generative-drifting-model-trajectories"
out = figure_dir(NAME)
rng = np.random.default_rng(44)

We compare unnormalized and normalized Laplacian-kernel drifts on the same particles.  The integration is intentionally small but long enough for the normalized trajectories to reach the two target modes.

In [2]:
def cloud(center, cov, n):
    return rng.multivariate_normal(center, cov, n)

x0 = np.vstack([
    cloud([-1.15, -0.40], [[0.035, 0.005], [0.005, 0.045]], 12),
    cloud([-1.00,  0.55], [[0.035, -0.006], [-0.006, 0.035]], 12),
])
y = np.vstack([
    cloud([0.72, -0.52], [[0.030, 0.006], [0.006, 0.028]], 12),
    cloud([1.05,  0.48], [[0.026, -0.004], [-0.004, 0.036]], 12),
])
n = len(x0)

def B_norm(X, Y, eps=0.18):
    diff = Y[None, :, :] - X[:, None, :]
    K = np.exp(-np.sum(diff**2, axis=-1) / (2*eps))
    return (K[:, :, None] * diff).sum(axis=1) / (K.sum(axis=1, keepdims=True) + 1e-12)

def B_raw(X, Y, eps=0.18):
    diff = Y[None, :, :] - X[:, None, :]
    K = np.exp(-np.sum(diff**2, axis=-1) / (2*eps))
    return (K[:, :, None] * diff).sum(axis=1) / len(Y)

def simulate(field, steps=78, dt=0.18, cap=1.75):
    traj = np.zeros((steps+1, n, 2))
    traj[0] = x0
    X = x0.copy()
    for k in range(steps):
        V = field(X)
        speed = np.linalg.norm(V, axis=1).max()
        if speed > 1e-12:
            V = V / max(speed, cap)
        X = X + dt * V
        traj[k+1] = X
    return traj

traj_raw = simulate(lambda X: 4.6 * (B_raw(X, y, eps=0.23) - 0.70 * B_raw(X, X, eps=0.20)), steps=78, dt=0.18, cap=2.0)
traj_norm = simulate(lambda X: 1.35 * (B_norm(X, y, eps=0.23) - 0.78 * B_norm(X, X, eps=0.18)), steps=86, dt=0.20, cap=1.65)
all_points = np.vstack([traj_raw.reshape(-1,2), traj_norm.reshape(-1,2), y])
xlim, ylim = padded_limits(all_points, pad=0.12)

In [3]:
def draw_traj(traj, filename):
    fig, ax = plt.subplots(figsize=(2.25, 1.98))
    for i in range(n):
        pts = traj[:, i]
        segs = np.stack([pts[:-1], pts[1:]], axis=1)
        cols = [(*interp_color(k/(len(pts)-2)), 0.37) for k in range(len(pts)-1)]
        ax.add_collection(LineCollection(segs, colors=cols, linewidths=0.56, zorder=1))
    ax.scatter(y[:,0], y[:,1], s=DIRAC_MARKER_SIZE * 0.50, marker="o", color=BLUE, edgecolor="none", linewidth=0, alpha=0.58, zorder=2)
    ax.scatter(traj[0,:,0], traj[0,:,1], s=DIRAC_MARKER_SIZE * 0.48, marker="o", color=RED, edgecolor="none", linewidth=0, zorder=3)
    ax.scatter(traj[-1,:,0], traj[-1,:,1], s=DIRAC_MARKER_SIZE * 0.54, marker="o", color=VIOLET, edgecolor="none", linewidth=0, zorder=4)
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_aspect("equal")
    remove_axes(ax)
    save_pdf(fig, out / filename, pad_inches=0.055)
    plt.close(fig)

draw_traj(traj_raw, "unnormalized-trajectories.pdf")
draw_traj(traj_norm, "normalized-trajectories.pdf")
# Backward-compatible alias.
draw_traj(traj_norm, "trajectories.pdf")

fig, ax = plt.subplots(figsize=(2.25, 1.98))
mid = traj_norm[len(traj_norm)//2]
grid_x = np.linspace(xlim[0], xlim[1], 13)
grid_y = np.linspace(ylim[0], ylim[1], 11)
GX, GY = np.meshgrid(grid_x, grid_y)
G = np.column_stack([GX.ravel(), GY.ravel()])
V = 1.35 * (B_norm(G, y, eps=0.23) - 0.78 * B_norm(G, mid, eps=0.18))
Vnorm = np.linalg.norm(V, axis=1, keepdims=True)
V = V / (Vnorm + 1e-12) * np.minimum(Vnorm, 0.22)
ax.quiver(G[:,0], G[:,1], V[:,0], V[:,1], angles="xy", scale_units="xy", scale=1.0, color=GRAY, width=0.0045, alpha=0.52, zorder=1)
ax.scatter(mid[:,0], mid[:,1], s=DIRAC_MARKER_SIZE * 0.50, marker="o", color=VIOLET, edgecolor="none", linewidth=0, zorder=3)
ax.scatter(y[:,0], y[:,1], s=DIRAC_MARKER_SIZE * 0.50, marker="o", color=BLUE, edgecolor="none", linewidth=0, alpha=0.65, zorder=2)
ax.set_xlim(*xlim)
ax.set_ylim(*ylim)
ax.set_aspect("equal")
remove_axes(ax)
save_pdf(fig, out / "normalized-field.pdf", pad_inches=0.055)
save_pdf(fig, out / "field.pdf", pad_inches=0.055)
plt.close(fig)